# Morphology Tower — MobileNetV3 Training (Task 1: Days 1–6)

**Goal:** build the visual classification tower that labels particle morphology:
`sphere / fragment / fiber / film / foam`.

**Pipeline in this notebook**

1. Mount Google Drive (dataset lives at `microplastic-ipd/morphology`)
2. Install deps (`timm`, `albumentations`, ...)
3. Build augmentations  → becomes `src/preprocessing/image_preprocess.py`
4. Build `VisionTower`   → becomes `src/models/vision_tower.py`
5. DataLoaders + training loop → becomes `src/training/train_morphology.py`
6. **Run 1** (epochs 1–10): backbone frozen, head only, `NAdam(lr=1e-3)`
7. **Run 2** (epochs 11–30): unfreeze last 4 blocks, diff lr (1e-4 / 1e-3)
8. **Run 3** (epochs 31–50): full fine-tune `lr=1e-5`
9. Final test eval: **macro-F1 + per-class F1 + confusion matrix** →
   `results/classification/confusion_morphology.png`

**Before running**

- Dataset uploaded to Drive at `MyDrive/microplastic-ipd/morphology`
  (train / val / test / class_weights.json / splits.json)
- Runtime → Change runtime type → **T4 GPU**
- Expected wall time on T4: ~1.5–3 h for all 50 epochs.

**After running**: keep the notebook — it is the single source of truth. Cell
sections marked `MODULE → src/...` map 1:1 to `.py` files for integration.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


## 1. Install dependencies

`torch` / `torchvision` are preinstalled on Colab. We add `timm`, `albumentations`,
`seaborn`, `grad-cam`, `wandb`. (wandb is optional; set `use_wandb=False` below to skip.)

In [ ]:

!pip install -q -U timm albumentations seaborn grad-cam wandb
import torch, torchvision
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
import timm, albumentations
print('timm', timm.__version__)
print('albumentations', albumentations.__version__)


## 2. Configuration + data location (Drive)

The notebook locates the dataset at `/content/drive/MyDrive/microplastic-ipd/morphology`
(the folder you uploaded). It falls back to a couple of alternate paths, then prints a
class-count summary (mini-EDA) and loads `class_weights.json`.

In [ ]:

import json, os, random, shutil
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, confusion_matrix

CONFIG = {
    "drive_base": "/content/drive/MyDrive/microplastic-ipd",
    "classes": ["sphere", "fragment", "fiber", "film", "foam"],
    "img_size": 224,
    "batch_size": 32,
    "num_workers": 2,
    "seed": 42,
    # Runs (epochs are 1-indexed cumulative)
    "run1_epochs": 10,     # epochs 1-10 frozen
    "run2_epochs": 20,     # epochs 11-30 unfreeze last 4 blocks
    "run3_epochs": 20,     # epochs 31-50 full fine-tune
    "lr_head": 1e-3,
    "lr_backbone": 1e-4,
    "lr_full": 1e-5,
    "weight_decay": 1e-4,
    "unfreeze_blocks": 4,
    "use_amp": True,
    "use_wandb": False,
    "do_run1": True,
    "do_run2": True,
    "do_run3": True,
}

def find_data_dir() -> Path:
    base = Path(CONFIG["drive_base"])
    cands = [
        base / "morphology",
        Path("/content/drive/MyDrive/morphology"),
    ]
    for c in cands:
        if (c / "train").is_dir() and (c / "val").is_dir() and (c / "test").is_dir():
            return c
    raise RuntimeError(
        "Morphology dataset not found on Drive. Expected: "
        "MyDrive/microplastic-ipd/morphology with train/val/test folders. "
        "Upload it (1.4 GB) and re-run this cell."
    )

DATA_DIR = find_data_dir()
print("DATA_DIR:", DATA_DIR)

# seeds
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

# quick class distribution (mini-EDA)
for split in ["train", "val", "test"]:
    counts = {}
    for cls in CONFIG["classes"]:
        d = DATA_DIR / split / cls
        if d.is_dir():
            counts[cls] = len([f for f in os.listdir(d) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    print(f"{split:5s}", counts)

# class weights (inverse frequency, produced by download_data.py organize)
with open(DATA_DIR / "class_weights.json") as fh:
    class_weights = json.load(fh)
print("class_weights:", class_weights)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 3. MODULE → `src/preprocessing/image_preprocess.py`

Albumentations pipelines:
- **train:** HFlip / VFlip / RandomRotate90 (p=0.5) + ColorJitter + ImageNet normalize
- **val/test:** Resize only + normalize (no augmentation)

In [ ]:

import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_train_transform(size: int = 224):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.Resize(size, size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def get_eval_transform(size: int = 224):
    return A.Compose([
        A.Resize(size, size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


# sanity check: random uint8 image -> (3, size, size) float tensor
import numpy as _np
_dummy = _np.random.randint(0, 255, (300, 400, 3), dtype=_np.uint8)
_out = get_eval_transform()(image=_dummy)["image"]
print("transform output shape:", tuple(_out.shape), "dtype:", _out.dtype)
assert tuple(_out.shape) == (3, 224, 224), "transform shape mismatch"


## 4. MODULE → `src/models/vision_tower.py`

`VisionTower(nn.Module)` — timm `mobilenetv3_small_100` (pretrained, `num_classes=0`,
`global_pool='avg'` → **1024 features**; the feature dim is read from the model at
runtime, never hardcoded) + projection head
`Linear(1024→256) → LayerNorm → ReLU → Linear(256→5)`.

Includes freeze/unfreeze helpers for the 3 training runs.

In [ ]:

import torch
import torch.nn as nn
import timm


class VisionTower(nn.Module):
    '''MobileNetV3-Small morphology classifier (sphere/fragment/fiber/film/foam).'''

    def __init__(self, num_classes: int = 5, pretrained: bool = True) -> None:
        super().__init__()
        self.num_classes = num_classes
        # global_pool='avg' + num_classes=0 → flat feature vector (1024 for small)
        self.backbone = timm.create_model(
            "mobilenetv3_small_100",
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )
        feat_dim = getattr(self.backbone, "num_features", None)
        if feat_dim is None:  # fallback: measure empirically, timm-version-proof
            with torch.no_grad():
                feat_dim = self.backbone(torch.zeros(1, 3, 224, 224)).shape[1]
        self.feat_dim = feat_dim
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )
        if pretrained:
            self.freeze_backbone(freeze=True)

    def freeze_backbone(self, freeze: bool = True) -> None:
        for p in self.backbone.parameters():
            p.requires_grad = not freeze

    def unfreeze_last_blocks(self, n: int = 4) -> None:
        blocks = list(self.backbone.blocks)
        for blk in blocks[-n:]:
            for p in blk.parameters():
                p.requires_grad = True

    def unfreeze_all(self) -> None:
        self.freeze_backbone(freeze=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(x)          # [B, feat_dim]
        return self.head(feats)           # [B, 5]


# sanity check: [1,3,224,224] -> [1,5]
model = VisionTower(num_classes=5, pretrained=True).to(device)
model.eval()
with torch.no_grad():
    out = model(torch.randn(1, 3, 224, 224).to(device))
print("forward output shape:", tuple(out.shape))
assert tuple(out.shape) == (1, 5), "vision tower output shape mismatch"


## 5. MODULE → `src/training/train_morphology.py` (dataset + loaders)

Reads the ImageFolder-style layout `data/processed/morphology/{train,val,test}/<class>/`
using the albumentations transforms defined above.

In [ ]:

import torch
from PIL import Image


class MorphologyDataset(torch.utils.data.Dataset):
    '''ImageFolder-style reader for morphology splits.'''

    def __init__(self, root, classes, transform):
        super().__init__()
        self.samples = []
        self.classes = classes
        for label, cls in enumerate(classes):
            cdir = os.path.join(root, cls)
            if not os.path.isdir(cdir):
                print("  [warn] missing class dir:", cdir)
                continue
            for name in sorted(os.listdir(cdir)):
                if name.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append((os.path.join(cdir, name), label))
        self.transform = transform
        print(f"  {os.path.basename(str(root))}: {len(self.samples)} images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = np.array(Image.open(path).convert("RGB"))
        image = self.transform(image=image)["image"]
        return image, label


def build_loaders(data_dir, config):
    train_ds = MorphologyDataset(
        os.path.join(data_dir, "train"), config["classes"], get_train_transform(config["img_size"])
    )
    val_ds = MorphologyDataset(
        os.path.join(data_dir, "val"), config["classes"], get_eval_transform(config["img_size"])
    )
    test_ds = MorphologyDataset(
        os.path.join(data_dir, "test"), config["classes"], get_eval_transform(config["img_size"])
    )
    kw = dict(
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
        pin_memory=True,
        persistent_workers=config["num_workers"] > 0,
    )
    train_loader = torch.utils.data.DataLoader(train_ds, shuffle=True, drop_last=True, **kw)
    val_loader = torch.utils.data.DataLoader(val_ds, shuffle=False, **kw)
    test_loader = torch.utils.data.DataLoader(test_ds, shuffle=False, **kw)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_loaders(DATA_DIR, CONFIG)

# class-weight loss tensor (aligned with CONFIG["classes"])
weight_tensor = torch.tensor(
    [float(class_weights[c]) for c in CONFIG["classes"]], dtype=torch.float32, device=device
)
print("loss weights:", weight_tensor.tolist())


## 6. MODULE → `src/training/train_morphology.py` (training loop)

Shared loop with 3 run modes (optimizer groups + freeze state):

- `run1`: backbone frozen, head only           NAdam(head, lr=1e-3)
- `run2`: last 4 blocks unfrozen               NAdam([blocks 1e-4, head 1e-3])
- `run3`: everything unfrozen                  NAdam(all, lr=1e-5)

Tracks a `history` dict across runs, saves a best checkpoint by val macro-F1 to
`weights/mobilenetv3_morphology_best.pth`, and a per-run checkpoint for resume.

In [ ]:

from tqdm.auto import tqdm
import torch.nn as nn


CRITERION = nn.CrossEntropyLoss(weight=weight_tensor)


def build_optimizer(model, config, mode):
    wd = config["weight_decay"]
    if mode == "run1":
        params = [p for p in model.parameters() if p.requires_grad]
        return torch.optim.NAdam(params, lr=config["lr_head"], weight_decay=wd)
    if mode == "run2":
        model.unfreeze_last_blocks(config["unfreeze_blocks"])
        backbone_params = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        print(f"  unfroze last {config['unfreeze_blocks']} blocks "
              f"(backbone params {len(backbone_params)}, head params {len(head_params)})")
        return torch.optim.NAdam(
            [
                {"params": backbone_params, "lr": config["lr_backbone"]},
                {"params": head_params, "lr": config["lr_head"]},
            ],
            weight_decay=wd,
        )
    # run3
    model.unfreeze_all()
    return torch.optim.NAdam(model.parameters(), lr=config["lr_full"], weight_decay=wd)


def train_one_epoch(model, loader, criterion, optimizer, config):
    model.train()
    total, correct = 0, 0
    running = 0.0
    scaler = torch.cuda.amp.GradScaler(enabled=config["use_amp"])
    pbar = tqdm(loader, desc="train", leave=False)
    for images, targets in pbar:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=config["use_amp"]):
            logits = model(images)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        total += targets.size(0)
        correct += (preds == targets).sum().item()
        pbar.set_postfix(loss=loss.item())
    return running / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def validate(model, loader, criterion, config):
    model.eval()
    total, correct = 0, 0
    running = 0.0
    all_preds, all_labels = [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        with torch.autocast(device_type=device.type, enabled=config["use_amp"]):
            logits = model(images)
            loss = criterion(logits, targets)
        running += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        total += targets.size(0)
        correct += (preds == targets).sum().item()
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(targets.cpu().tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return running / max(total, 1), correct / max(total, 1), macro_f1


def save_checkpoint(model, path, epoch, val_macro_f1):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_macro_f1": val_macro_f1,
            "classes": CONFIG["classes"],
        },
        str(path),
    )


def load_checkpoint(model, path):
    ck = torch.load(str(path), map_location=device)
    model.load_state_dict(ck["model_state_dict"])
    return ck.get("epoch", 0), ck.get("val_macro_f1", 0.0)


def train_run(model, train_loader, val_loader, config, mode, start_epoch, n_epochs,
              run_name, history, resume_from=None):
    '''Train `n_epochs` starting at cumulative epoch `start_epoch`.'''
    if resume_from and Path(resume_from).exists():
        ep, prev_f1 = load_checkpoint(model, resume_from)
        print(f"[{run_name}] resuming from cached checkpoint (epoch {ep}, val_macro_f1 {prev_f1:.4f})")
        return history

    optimizer = build_optimizer(model, config, mode)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    best_f1 = history.get("best_val_macro_f1", 0.0)

    print(f"=== {run_name}: epochs {start_epoch}-{start_epoch + n_epochs - 1} [mode={mode}] ===")
    for i in range(n_epochs):
        epoch = start_epoch + i
        train_loss, train_acc = train_one_epoch(model, train_loader, CRITERION, optimizer, config)
        val_loss, val_acc, val_f1 = validate(model, val_loader, CRITERION, config)
        scheduler.step()

        history.setdefault("epoch", []).append(epoch)
        history.setdefault("train_loss", []).append(train_loss)
        history.setdefault("val_loss", []).append(val_loss)
        history.setdefault("val_acc", []).append(val_acc)
        history.setdefault("val_macro_f1", []).append(val_f1)

        print(f"epoch {epoch:02d} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | "
              f"val_loss {val_loss:.4f} | val_acc {val_acc:.4f} | val_macro_f1 {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            save_checkpoint(model, "weights/mobilenetv3_morphology_best.pth", epoch, val_f1)
            print(f"  -> saved best checkpoint (val_macro_f1 {val_f1:.4f})")

        if config["use_wandb"]:
            wandb.log({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                       "val_acc": val_acc, "val_macro_f1": val_f1})

    if config["use_wandb"]:
        wandb.log({"run_finished": run_name})

    save_checkpoint(model, f"weights/{run_name}.pth", start_epoch + n_epochs - 1, best_f1)
    history["best_val_macro_f1"] = best_f1
    return history


## 7. Run 1 — frozen backbone (epochs 1–10)

Only the projection head trains at `lr=1e-3`. Expected frozen-backbone macro-F1
≈ 0.65–0.75 (Guide Day 4). This is your transfer-learning baseline.

In [ ]:

history = {}

if CONFIG["use_wandb"]:
    import wandb
    wandb.init(project="microplastic-ipd", name="morphology-tower", config=CONFIG)

model = VisionTower(num_classes=5, pretrained=True).to(device)

if CONFIG["do_run1"]:
    history = train_run(
        model, train_loader, val_loader, CONFIG, mode="run1",
        start_epoch=1, n_epochs=CONFIG["run1_epochs"],
        run_name="run1", history=history,
        resume_from="weights/run1.pth",
    )
else:
    print("run1 skipped (do_run1=False)")

print("frozen-backbone macro-F1 (epoch 10):",
      history.get("val_macro_f1", [0])[-1] if history else "n/a")


## 8. Run 2 — progressive unfreezing (epochs 11–30)

Unfreezes the last 4 backbone blocks; differential LR: blocks `1e-4`, head `1e-3`.
Continues from Run 1's trained weights.

In [ ]:

if CONFIG["do_run2"]:
    history = train_run(
        model, train_loader, val_loader, CONFIG, mode="run2",
        start_epoch=CONFIG["run1_epochs"] + 1, n_epochs=CONFIG["run2_epochs"],
        run_name="run2", history=history,
        resume_from="weights/run2.pth",
    )
else:
    print("run2 skipped (do_run2=False)")


## 9. Run 3 — full fine-tune (epochs 31–50)

All layers, `lr=1e-5`, cosine schedule. (Guide Day 6 final phase.)

In [ ]:

if CONFIG["do_run3"]:
    history = train_run(
        model, train_loader, val_loader, CONFIG, mode="run3",
        start_epoch=CONFIG["run1_epochs"] + CONFIG["run2_epochs"] + 1,
        n_epochs=CONFIG["run3_epochs"],
        run_name="run3", history=history,
        resume_from="weights/run3.pth",
    )
else:
    print("run3 skipped (do_run3=False)")

if CONFIG["use_wandb"]:
    wandb.finish()


## 10. Final evaluation — test split + confusion matrix

Loads the best checkpoint (by val macro-F1), evaluates the **held-out test split**,
prints per-class F1, and writes `results/classification/confusion_morphology.png`.
This is the issue's named deliverable.

In [ ]:

model = VisionTower(num_classes=5, pretrained=True).to(device)
ck_path = Path("weights/mobilenetv3_morphology_best.pth")
if ck_path.exists():
    load_checkpoint(model, ck_path)
    print("loaded best checkpoint:", ck_path)
else:
    print("!! best checkpoint not found - evaluating the in-memory model !!")

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="test"):
        images = images.to(device)
        with torch.autocast(device_type=device.type, enabled=CONFIG["use_amp"]):
            logits = model(images)
        preds = logits.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(targets.tolist())

acc = (np.array(all_preds) == np.array(all_labels)).mean()
macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)

print(f"\n=== TEST RESULTS ===")
print(f"accuracy : {acc:.4f}")
print(f"macro-F1 : {macro_f1:.4f}\n")
print(f"{'class':10s} {'F1':>8s} {'support':>8s}")
for i, cls in enumerate(CONFIG["classes"]):
    support = all_labels.count(i)
    print(f"{cls:10s} {per_class_f1[i]:8.4f} {support:8d}")

report = {
    "accuracy": float(acc),
    "macro_f1": float(macro_f1),
    "per_class_f1": {cls: float(per_class_f1[i]) for i, cls in enumerate(CONFIG["classes"])},
}

cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(CONFIG["classes"]))))
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CONFIG["classes"], yticklabels=CONFIG["classes"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Morphology Confusion Matrix")
plt.tight_layout()
os.makedirs("results/classification", exist_ok=True)
plt.savefig("results/classification/confusion_morphology.png", dpi=150)
plt.show()
print("saved results/classification/confusion_morphology.png")


## 11. Training curves + save outputs to Drive

Plots loss / val macro-F1 across all epochs and copies every artifact to
`MyDrive/microplastic-ipd/outputs/` so the Colab session can be thrown away safely.

In [ ]:
os.makedirs("results/classification", exist_ok=True)

if history.get("epoch"):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history["epoch"], history["train_loss"], label="train")
    ax[0].plot(history["epoch"], history["val_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].legend(); ax[0].set_xlabel("epoch")
    ax[1].plot(history["epoch"], history["val_macro_f1"], label="val macro-F1", color="tab:orange")
    ax[1].set_title("Macro-F1"); ax[1].legend(); ax[1].set_xlabel("epoch")
    plt.tight_layout()
    plt.savefig("results/classification/training_curves.png", dpi=150)
    plt.show()
else:
    print("no history to plot (all runs resumed/off?)")

os.makedirs("results/classification", exist_ok=True)
# persist everything to Drive (next to the morphology folder that was uploaded)
drive_out = Path(DATA_DIR).parent / "outputs"
for rel in [
    "weights/mobilenetv3_morphology_best.pth",
    "weights/run1.pth",
    "weights/run2.pth",
    "weights/run3.pth",
    "results/classification/confusion_morphology.png",
    "results/classification/training_curves.png",
]:
    src = Path(rel)
    if src.exists():
        dst = drive_out / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(src), str(dst))
        print("copied ->", dst)

# also save the per-class report JSON
with open("results/classification/per_class_f1.json", "w") as fh:
    json.dump(report, fh, indent=2)
dst = drive_out / "results/classification/per_class_f1.json"
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2("results/classification/per_class_f1.json", str(dst))
print("copied ->", dst)

print("\n=== DRIVE OUTPUTS ===")
for f in sorted(drive_out.rglob("*")):
    if f.is_file():
        print(" ", f.relative_to(drive_out), f"{f.stat().st_size/1e6:.1f} MB")


## 12. Hand back after running

1. **Download this notebook** (File → Download / ⬇ icon) once it has run to completion.
2. Also keep everything under `MyDrive/microplastic-ipd/outputs/` (weights, confusion
   matrix, curves, per-class report).
3. Hand the downloaded `.ipynb` back — the `MODULE → src/...` sections map 1:1 to the
   `.py` files that get committed for integration:

| Notebook section | Target `.py` |
|---|---|
| §3 augmentations | `src/preprocessing/image_preprocess.py` |
| §4 `VisionTower` | `src/models/vision_tower.py` |
| §5 dataset/loaders | `src/training/train_morphology.py` |
| §6 training loop | `src/training/train_morphology.py` |
| §10 eval + confusion | `src/training/metrics.py` + CLI in `train_morphology.py` |

**Note on class imbalance:** Film (35) and Foam (8) are heavily under-represented
(class weights 34.3 and 150.2). Report the honest macro-F1; per-class F1 and the
confusion matrix tell the real story.